## 导库

没有安装以下库的话，请先安装

In [23]:
# !pip install jieba
# !pip install plotly
# !pip install torch

In [24]:
import re
import torch
import torch.nn as nn
import numpy as np
import random
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader, RandomSampler
import torch.nn.functional as F
import torch.optim as optim
import time
import math
import jieba
import plotly.express as px
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### 工具函数

In [25]:
def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)


def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

## 数据预处理

为了不影响准确度，如果字符属于数字，就将其替换为数字特殊 token-
“[num]”，处理后句子变成“[num]年”；

如果字符属于其他语言，就将其替换为字母特殊 token-“[eng]”，处理后句子变成“[eng]”；


In [26]:
filtering_data_file = './data/MuCGEC_dev.txt'
filtered_data_file = './data/MuCGEC_dev_filtered.txt'

In [27]:
with open(filtering_data_file, 'r') as f:
    lines = f.readlines()
    # 去掉最前面的数字和制表符
    sentences = [line.split('\t', 1)[1] for line in lines]
    sentences = [sentence.split('\t') for sentence in sentences]
    eng_pattern = re.compile(r'[a-zA-Z]+')
    num_pattern = re.compile(r'\d+')
    # 去掉最后的换行符
    for i, sentence in enumerate(sentences):
        for j, s in enumerate(sentence):

            s = s.strip('\n').strip('。')
            # 使用sub方法将非中文部分替换为[eng]
            eng_found = eng_pattern.findall(s)
            for eng in eng_found:
                s = s.replace(eng, '[eng]')
            num_found = num_pattern.findall(s)
            for num in num_found:
                s = s.replace(num, '[num]')
            # 将非中文部分的.替换为。
            sentence[j] = s.replace('.', '。').replace('?', '？').replace(
                '!', '！').replace(';', '；').replace(',', '，')

        sentences[i] = sentence
print(sentences[:10])

[['因为在冰箱里没什么东西也做很好吃的菜', '即使在冰箱里没什么东西也能做很好吃的菜', '即使在冰箱里没什么东西，也能做很好吃的菜'], ['再说，他们可能以为自己吸烟时对别人带来的不便和不舒服也不可能很大', '再说，他们可能以为自己吸烟时给别人带来的不便和不舒服也不会很大', '再说，他们可能以为自己吸烟时给别人带来的不便和不舒服也不是很大', '再说，他们可能以为自己吸烟时给别人带来的不便和不舒服也不可能很大'], ['这样我们在生活当中没有了跟男生接触的机会，总会有一天离开学校的，我们那时肯定会遇到到底怎样与男人交流、沟通的问题了', '这样我们在生活当中没有了跟男生接触的机会，但我们总会有一天要离开学校的，我们离开学校后肯定会遇到到底应该怎样与男人交流、沟通的问题', '这样我们在生活当中没有了跟男生接触的机会，等有一天离开学校的时候，我们肯定会遇到到底应该怎样与男人交流、沟通的问题', '这样我们在生活当中没有了跟男生接触的机会，我们总会有一天离开学校的，我们那时肯定会遇到到底怎样与男人交流、沟通的问题'], ['我不愿意这么过分的喜欢歌手', '我不愿意这么过分地喜欢歌手', '我不愿意这么过分痴迷地喜欢歌手'], ['学生大概做飞机去北京', '学生大概坐飞机去北京', '学生大概是坐飞机去北京'], ['今天我看见了我家周边的西药店，吊着牌子称；“口罩都卖光了”', '今天我看见我家周边的西药店，吊着牌子称：“口罩都卖光了。”', '今天我看见我家周边的西药店吊着牌子称：“口罩都卖光了。”'], ['我是中学生，不知因为我是喜爱流行、时尚的人，但我认为流行歌曲是一种自己享受快乐的一种方法', '我是中学生，不知是否是因为我是一个喜爱流行、时尚的人才会这么想，但我确实认为流行歌曲是一种自己享受快乐的方法', '我是中学生，不知是否是因为我是一个喜爱流行、时尚的人才会这么想，但我确实认为流行歌曲是自己享受快乐的一种方法'], ['改变自己的身体的冲动往往是一个深刻精神不稳定的症状。它应该被视为一个问题，而不是纵容还有鼓励美容手术治疗', '改变自己身体的冲动往往是一种深层次的精神不稳定的症状。它应该被视为是一个问题，而不是纵容、鼓励美容手术治疗', '改变自己的身体的冲动往往是一个精神极不稳定的症状。它应该被视为一个问题，而不是被纵容或者被鼓励采

每一个错误句子只能有一个正确句子  
如果一行只有一个句子，认为他是正确的，对应的正确句也就是他  
如果一行有多个句子，认为有多个正确句，遍历正确句，每一个正确句和错误句组成一个样本


In [28]:
filtered_sentences = []
for index, sentence in enumerate(sentences):
    if len(sentence) == 1:
        filtered_sentences.append([sentence[0], sentence[0]])
        continue
    elif len(sentence) == 2:
        if sentence[1] == '没有错误':
            sentence[1] = sentence[0]
        filtered_sentences.append(sentence)
        continue
    elif len(sentence) > 2:
        for i in range(1, len(sentence)):
            filtered_sentences.append([sentence[0], sentence[i]])
        filtered_sentences.pop(index)
    elif sentence[1].find('[缺失成分]') or sentence[1].find('[缺失成分]'):
        filtered_sentences.pop(index)
print(filtered_sentences[:10])

[['因为在冰箱里没什么东西也做很好吃的菜', '即使在冰箱里没什么东西，也能做很好吃的菜'], ['再说，他们可能以为自己吸烟时对别人带来的不便和不舒服也不可能很大', '再说，他们可能以为自己吸烟时给别人带来的不便和不舒服也不是很大'], ['这样我们在生活当中没有了跟男生接触的机会，总会有一天离开学校的，我们那时肯定会遇到到底怎样与男人交流、沟通的问题了', '这样我们在生活当中没有了跟男生接触的机会，但我们总会有一天要离开学校的，我们离开学校后肯定会遇到到底应该怎样与男人交流、沟通的问题'], ['这样我们在生活当中没有了跟男生接触的机会，总会有一天离开学校的，我们那时肯定会遇到到底怎样与男人交流、沟通的问题了', '这样我们在生活当中没有了跟男生接触的机会，我们总会有一天离开学校的，我们那时肯定会遇到到底怎样与男人交流、沟通的问题'], ['我不愿意这么过分的喜欢歌手', '我不愿意这么过分痴迷地喜欢歌手'], ['学生大概做飞机去北京', '学生大概是坐飞机去北京'], ['今天我看见了我家周边的西药店，吊着牌子称；“口罩都卖光了”', '今天我看见我家周边的西药店吊着牌子称：“口罩都卖光了。”'], ['我是中学生，不知因为我是喜爱流行、时尚的人，但我认为流行歌曲是一种自己享受快乐的一种方法', '我是中学生，不知是否是因为我是一个喜爱流行、时尚的人才会这么想，但我确实认为流行歌曲是自己享受快乐的一种方法'], ['改变自己的身体的冲动往往是一个深刻精神不稳定的症状。它应该被视为一个问题，而不是纵容还有鼓励美容手术治疗', '改变自己的身体的冲动往往是一个精神极不稳定的症状。它应该被视为一个问题，而不是被纵容或者被鼓励采用美容手术治疗'], ['这个巨大飞的物体停住了一会就又向天空快速地飞去。一闪而过，变成了一个形影一样的亮点，一点点地暗下去', '这个巨大的飞行物体停了一会儿就又向天空快速地飞去了。它一闪而过，变成了一个星星一样的亮点，然后一点点地暗下去']]


写入文件**filtered.txt中

In [29]:
with open(filtered_data_file, 'w') as f:
    for sentence in filtered_sentences:
        f.write(sentence[0] + ' ' + sentence[1] + '\n')

## 加载数据

In [30]:
train_data_file = './data/MuCGEC_dev_filtered.txt'  # 训练数据
test_data_file = './data/MuCGEC_test_filtered.txt'

### 辅助加载类

这个用来存储每一个词语的索引和单词哈希表

In [31]:
SOS_token = 0
EOS_token = 1


class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "[SOS]", 1: "[EOS]"}
        # self.index2word = {}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in jieba.lcut(sentence):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

根据每段句子的长度，确定最大长度

In [32]:
with open(train_data_file, 'r') as f:
    lines = f.readlines()
    sentences = [[s for s in l.split(' ')] for l in lines]
# 计算每个句子的长度
sentence_lengths = [len(jieba.lcut(sentence_pair[0]))
                    for sentence_pair in sentences]

# 使用Plotly绘制直方图
fig = px.histogram(sentence_lengths, nbins=200, title='Sentence Length Distribution', labels={
                   'value': 'Sentence Length', 'count': 'Frequency'})
fig.show()

图中句子的最大长度在40-60之间更好

将数据集修剪为相对较短且简单的句子

In [47]:
MAX_LENGTH = 40


def filterPair(p):
    # 因为后面会加<EOS>，所以这里减去一些长度
    ret = len(jieba.lcut(p[0])) < MAX_LENGTH and len(
        jieba.lcut(p[1])) < MAX_LENGTH
    return ret

In [48]:
def prepareData(data_file=train_data_file):

    input_lang = Lang('input')
    output_lang = Lang('output')
    pairs = []

    with open(data_file, 'r') as f:
        lines = f.readlines()
        for line in lines:
            pair = line.strip('\n').split(' ')
            if not filterPair(pair):
                # 如果长度不符合要求则跳过
                continue
            pairs.append(pair)
            input_lang.addSentence(pair[0])
            output_lang.addSentence(pair[1])
    return input_lang, output_lang, pairs

### 准备数据

In [49]:
input_lang, output_lang, pairs = prepareData(data_file=train_data_file)
print("Counted words:")
print(input_lang.name, input_lang.n_words)
print(output_lang.name, output_lang.n_words)
print(random.choice(pairs))

Counted words:
input 3187
output 3217
['因为，孩子在生长的过程上很受父母的影响', '因为，孩子在成长的过程中很受父母影响']


## 构建模型

### 定义编码器

In [50]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.gru(embedded)
        return output, hidden

### 定义解码器

### 构建注意力机制

In [51]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size)
        self.Ua = nn.Linear(hidden_size, hidden_size)
        self.Va = nn.Linear(hidden_size, 1)

    def forward(self, query, keys):
        scores = self.Va(torch.tanh(self.Wa(query) + self.Ua(keys)))
        scores = scores.squeeze(2).unsqueeze(1)

        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights, keys)

        return context, weights


class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(AttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = BahdanauAttention(hidden_size)
        self.gru = nn.GRU(2 * hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(
            batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)
            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(
                    1)  # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                # detach from history as input
                decoder_input = topi.squeeze(-1).detach()

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions

    def forward_step(self, input, hidden, encoder_outputs):
        embedded = self.dropout(self.embedding(input))

        query = hidden.permute(1, 0, 2)
        context, attn_weights = self.attention(query, encoder_outputs)
        input_gru = torch.cat((embedded, context), dim=2)

        output, hidden = self.gru(input_gru, hidden)
        output = self.out(output)

        return output, hidden, attn_weights

## 训练模型

In [54]:
def indexesFromSentence(lang, sentence):
    # 将句子中的每个单词转换为对应的索引
    return [lang.word2index[word] for word in jieba.lcut(sentence)]


def tensorFromSentence(lang, sentence):
    # 将句子转换为索引的Tensor
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)


def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)


def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData()

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)

        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(
        train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

In [55]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
                decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(
            encoder_outputs, encoder_hidden, target_tensor)
        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [56]:
plt.switch_backend('agg')


def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [57]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
          print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)

    encoder.train()
    decoder.train()

    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder,
                           encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                         epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

### 训练启动

In [58]:
hidden_size = 256
batch_size = 64
n_epochs = 100

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder,
      n_epochs, print_every=1, plot_every=1)

0m 1s (- 1m 57s) (1 1%) 4.3627
0m 2s (- 1m 56s) (2 2%) 3.2184
0m 3s (- 1m 54s) (3 3%) 3.0776
0m 4s (- 1m 52s) (4 4%) 2.9688
0m 5s (- 1m 50s) (5 5%) 2.8618
0m 6s (- 1m 49s) (6 6%) 2.7494
0m 8s (- 1m 47s) (7 7%) 2.6367
0m 9s (- 1m 46s) (8 8%) 2.5181
0m 10s (- 1m 44s) (9 9%) 2.4004
0m 11s (- 1m 43s) (10 10%) 2.2770
0m 12s (- 1m 42s) (11 11%) 2.1539
0m 13s (- 1m 41s) (12 12%) 2.0308
0m 15s (- 1m 40s) (13 13%) 1.9132
0m 16s (- 1m 39s) (14 14%) 1.7883
0m 17s (- 1m 38s) (15 15%) 1.6712
0m 18s (- 1m 37s) (16 16%) 1.5556
0m 19s (- 1m 36s) (17 17%) 1.4445
0m 21s (- 1m 35s) (18 18%) 1.3384
0m 22s (- 1m 35s) (19 19%) 1.2340
0m 23s (- 1m 34s) (20 20%) 1.1376
0m 24s (- 1m 33s) (21 21%) 1.0488
0m 26s (- 1m 32s) (22 22%) 0.9608
0m 27s (- 1m 31s) (23 23%) 0.8797
0m 28s (- 1m 30s) (24 24%) 0.8035
0m 29s (- 1m 29s) (25 25%) 0.7322
0m 31s (- 1m 28s) (26 26%) 0.6669
0m 32s (- 1m 27s) (27 27%) 0.6057
0m 33s (- 1m 26s) (28 28%) 0.5512
0m 34s (- 1m 25s) (29 28%) 0.4999
0m 36s (- 1m 24s) (30 30%) 0.4548
0m 37s

## 评估模型

In [59]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(
            encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [60]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(
            encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence.replace(' ', '').replace('<EOS>', ''))
        print('')

### 评估启动

In [61]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> 对这样的措施吸烟者反对
= 对这样的措施吸烟者表示反对
< 对这样的措施吸烟者表示反对

> 孩子首先向父母学习应该怎么做人际关系、应该怎么学习等等
= 孩子首先向父母学习的应该是怎么做人、怎么处理人际关系、怎么学习等等
< 孩子首先向父母学习应该怎么做人、应该怎么处理人际关系、应该怎么处理人际关系、应该怎么学习等等、应该怎么处理人际关系、应该怎么学习等等、应该怎么处理人际关系、应该

> 在日本东京街头上能看到一些牌子。那牌子上写着“禁烟路”就是跟中国的措施一样。在街头上被发现抽烟的话，要罚款二千日元
= 在日本东京街头上能看到一些牌子，那牌子上写着“禁烟路”。就是跟中国的措施一样，在街头上被发现抽烟的话，要被罚款二千日元
< 在日本东京街头上能看到一些牌子。那牌子上写着“禁烟路”。就是跟中国的措施一样，在街头上被发现抽烟的话，要被罚款二千日元

> 综上所述，我们世界各国朋友们只有持之以恒坚持到底共同努力就会决解现在受苦挨饿的现象
= 综上所述，我们世界各国的朋友们只有持之以恒、坚持到底、共同努力才会改善现在受苦挨饿的现象
< 综上所述，我们世界各国的朋友们只有持之以恒、坚持到底、共同努力才会改善现在受苦挨饿的现象

> 总之，吸烟是不仅是对身体不好还会影响别人，所以如果以已经吸烟的人要吸烟时也不要在公共场所吸烟。如果还没吸烟的青少年请远离开烟
= 总之，吸烟不仅对身体不好，还会影响别人。所以吸烟的人如果要吸烟，也不要在公共场所吸；如果是还没吸过烟的青少年，请远离香烟
< 总之，吸烟不仅对身体不好，还会影响别人。所以吸烟的人如果要吸烟，也不要在公共场所吸；如果是还没吸过烟的青少年，请远离香烟

> 在公共场所抽烟是影响到他人的健康。每在别人抽烟的环境下留的一个小时，等于抽一条烟卷
= 在公共场所抽烟会影响到他人的健康。每在别人抽烟的环境中停留一个小时，等于抽一条烟卷
< 在公共场所抽烟是影响到他人的健康的行为。每在别人抽烟的环境下待一个小时，等于抽一条烟卷

> 还有吸烟对周围的人带来坏处，尤其是青少年与怀孕者
= 还有，吸烟会对周围的人带来不好的影响，尤其是青少年与怀孕者
< 还有吸烟给周围的人带来坏处，尤其是青少年与怀孕者的影响

> 冲绳是一个非常美丽的地方，天蓝蓝的，海水清淤见地，鱼特别新鲜，水果尝尝鲜。腓力和我夜夜下班以后，去了翘首美丽的星空
= 